# 실습과제 3 · 시계열 — 지하수위는 비 온 뒤 며칠 만에 오르나 (11주차 · 2026. 11. 16.)

기초지질데이터과학 · **제출: 오늘 23:59, LMS 실습과제 3 — 이 노트북(.ipynb) 하나, 각자 제출** · 5점 = 코드 2 · 해석 2 · 기록지 1

**이름 · 학번:**　　　　　　　　**쓴 AI:**

**자료** (둘 다 교육용 합성 자료 — 실제 관측 자료의 흔한 문제를 그대로 넣었습니다)
- `groundwater_hourly.csv` — 충적층 관측정 1곳, 2025년, **1시간 간격** 자동 로거. 열: `시각`(한국 시각), `수위_m`(표고, **클수록 지하수면이 높다**).
  현장 기록: 7월 초–중순 통신 장애 · 11/4 오전 로거 교체(새 로거의 기준 높이가 다를 수 있음) · 자료는 두 번에 나눠 내려받아 이어 붙였음.
- `rain_daily.csv` — 같은 유역의 강수 관측소, 2025년 **일강수량**. 기상청 자료와 같은 방식으로 **비가 오지 않은 날은 빈칸**.

**순서**: ① 원자료 점검(1번 칸) → ② 아래 프롬프트를 **그대로** AI에게 보내고(CSV 두 개 첨부) 답과 코드를 기록 → ③ 3번 칸의 도구로 AI의 처리를 **검증**하고, 틀린 곳을 고친 **내 최종 일 수위 `my_daily`**를 만든다 → ④ 해석 · 기록지.


## 0. 도구 — 실행만 하세요 (Shift + Enter)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
GW_FILE, RAIN_FILE = "groundwater_hourly.csv", "rain_daily.csv"

def load_raw():
    """원자료 두 개를 손대지 않고 읽는다."""
    return pd.read_csv(GW_FILE), pd.read_csv(RAIN_FILE)

def check_time(df, tcol, step):
    """시간 축 점검 — 순서 · 같은 시각 · 빠진 시각(행 자체가 없음) · 빈칸(값 없음). step: '1h' 또는 '1D'."""
    t = pd.to_datetime(df[tcol])
    print(f"행 {len(df):,} · 기간 {t.min()} ~ {t.max()}")
    print("시간 순서대로인가:", t.is_monotonic_increasing)
    print("같은 시각이 다시 나온 행:", int(t.duplicated().sum()))
    missing = pd.date_range(t.min(), t.max(), freq=step).difference(t)
    print("빠진 시각(행이 없음):", len(missing))
    if len(missing):
        m = missing.to_series()
        for _, g in m.groupby((m.diff() != pd.Timedelta(step)).cumsum()):
            print(f"   {g.iloc[0]} ~ {g.iloc[-1]}  ({len(g)}개)")
    print("빈칸(값 없음):", df.drop(columns=[tcol]).isna().sum().to_dict())

def as_hourly(gw):
    """원자료 → 시각 순 Series(같은 시각은 처음 값만). 빠진 시각은 빠진 채로 둔다."""
    s = gw.copy(); s["시각"] = pd.to_datetime(s["시각"])
    return s.drop_duplicates("시각").set_index("시각").sort_index()["수위_m"]

def jump_report(gw, limit=0.15):
    """이웃한 두 관측의 차이가 limit(m)보다 큰 곳. 스파이크는 올라갔다 곧 돌아오고, 단차는 그대로 남는다."""
    s = as_hourly(gw).dropna(); d = s.diff()
    rows = []
    for t in d.index[d.abs() > limit]:
        i = s.index.get_loc(t); t0 = s.index[i - 1]
        rows.append((t0, t, t - t0, s.iloc[i - 1], s.iloc[i], round(d[t], 3)))
    return pd.DataFrame(rows, columns=["앞 시각", "뒤 시각", "간격", "앞 값", "뒤 값", "변화(m)"])

def rain_series():
    """일강수량 Series — 빈칸은 비가 없었던 날이므로 0."""
    _, r = load_raw(); r["일시"] = pd.to_datetime(r["일시"])
    return r.set_index("일시")["일강수량(mm)"].fillna(0.0)

def rain_before(dates, days=7):
    """날짜마다 그날을 포함한 앞 (days+1)일의 강수 합 — 높은 수위가 비 때문인지 확인."""
    r = rain_series(); out = []
    for d in pd.to_datetime(pd.Index(dates)).normalize().unique():
        out.append((d.date(), round(r[d - pd.Timedelta(days=days): d].sum(), 1)))
    return pd.DataFrame(out, columns=["날짜", f"그날까지 {days + 1}일 강수(mm)"])

def before_after(s_hourly, when, hours=72):
    """when 앞뒤 hours시간의 중앙값 비교 — 로거 교체 같은 단차의 크기."""
    t = pd.Timestamp(when); s = s_hourly.sort_index()
    b = s[t - pd.Timedelta(hours=hours): t - pd.Timedelta(hours=1)].median()
    a = s[t: t + pd.Timedelta(hours=hours)].median()
    print(f"{when} 앞 {hours}시간 중앙값 {b:.3f} m · 뒤 {hours}시간 중앙값 {a:.3f} m · 차이 {a - b:+.3f} m")
    return round(a - b, 3)

def xcorr(level_daily, maxlag=10, plot=True):
    """강수(빈칸=0)와 일 수위의 교차상관 — '수위'와 '수위 변화량(오늘−어제)' 두 가지. 날짜로 맞추고 빈 날은 뺀다."""
    r = rain_series(); L = level_daily
    if isinstance(L, pd.DataFrame): L = L.select_dtypes("number").iloc[:, 0]
    L = L.copy(); L.index = pd.to_datetime(L.index); L = L[~L.index.duplicated()].sort_index().asfreq("D")
    cols = {}
    for name, y in [("수위", L), ("수위 변화량", L.diff())]:
        rs = []
        for k in range(maxlag + 1):
            yk = y.shift(-k).reindex(r.index); ok = yk.notna()
            rs.append(np.corrcoef(r[ok], yk[ok])[0, 1] if ok.sum() > 10 else np.nan)
        cols[name] = rs
    tab = pd.DataFrame(cols); tab.index.name = "시차(일)"
    if plot:
        ax = tab.rename(columns={"수위": "level", "수위 변화량": "daily change"}).plot(marker="o", figsize=(6, 3.2))
        ax.set_xlabel("lag (days): rain today -> level k days later"); ax.set_ylabel("correlation"); ax.grid(alpha=.3); plt.show()
    for c in tab: print(f"{c}: 상관이 가장 큰 시차 = {int(tab[c].idxmax())}일 (r = {tab[c].max():.2f})")
    return tab.round(3)

print("도구 준비 완료")


## 1. 원자료 첫 점검 (모두, 5분)

무엇이 이상한지 먼저 보고, AI가 그것을 어떻게 다루는지 지켜봅니다.

In [ ]:
gw, rain = load_raw()
print("=== groundwater_hourly.csv ==="); check_time(gw, "시각", "1h")
print("\n=== rain_daily.csv ==="); check_time(rain, "일시", "1D")
s = as_hourly(gw)
fig, ax = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
ax[0].plot(s.index, s.values, lw=.6); ax[0].set_ylabel("level (m)"); ax[0].set_ylim(44, 49)
r = rain_series(); ax[1].bar(r.index, r.values, width=1); ax[1].set_ylabel("rain (mm/day)")
plt.tight_layout(); plt.show()


## 2. AI에게 묻기 (10분)

아래 프롬프트를 **그대로** 보내고 CSV 두 개를 첨부하세요(첨부가 안 되면 내용을 붙여 넣기). 후속 질문은 자유지만, 처음 답은 꼭 기록합니다.

> 첨부한 지하수위 자료(groundwater_hourly.csv, 1시간 간격)와 강수 자료(rain_daily.csv)를 분석하는 파이썬 코드를 짜 줘. 지하수위를 하루 단위로 정리하면서 결측값과 이상값을 처리하고, 비가 온 뒤 며칠 만에 지하수위가 반응하는지 교차상관으로 구해 줘.

**AI의 답 기록** (아래를 채우세요)
- 빠진 시각 · 빈칸을 어떻게 처리했나:
- 중복된 시각은:
- 이상값은 무엇으로 찾았고(기준), 몇 개를 지웠나:
- 11/4 로거 교체는 다뤘나:
- 강수의 빈칸은 어떻게 처리했나:
- 반응 지연(교차상관 최대 시차)은 며칠이라고 했나:


In [ ]:
# AI가 준 코드를 여기에 붙여 실행하세요. 파일 이름은 그대로(groundwater_hourly.csv, rain_daily.csv).
# 실행 뒤, AI가 이상값으로 지운 날짜를 아래 목록에 적어 두면 3-2에서 확인할 수 있습니다. 예: ["2025-07-19", "2025-08-27"]
ai_removed_dates = []


## 3. 같은 기준에서 검증 (25분) — 칸마다 실행하고, AI의 처리와 비교하세요

In [ ]:
# 3-1. 시간 축 — 7/7–7/18(행이 없는 12일) 동안 비가 얼마나 왔나? 이 구간을 직선으로 이어도 될까?
r = rain_series()
print("7/7–7/18 강수 합:", r["2025-07-07":"2025-07-18"].sum(), "mm")
print(r["2025-07-07":"2025-07-18"][r["2025-07-07":"2025-07-18"] > 0])
s = as_hourly(gw)
print("\n빠진 구간 바로 앞 · 뒤 수위:", s[:"2025-07-06 23:00"].iloc[-1], "→", s["2025-07-19":].iloc[0], "m")


In [ ]:
# 3-2. 이상값 — 원자료에서 '이웃 시각과 0.15 m 넘게 다른 곳'. 스파이크는 쌍(올라감 → 돌아옴)으로, 단차는 하나만 나온다.
jumps = jump_report(gw)
display(jumps)
# AI가 지운 날짜가 실제로는 '큰 비 뒤의 상승'이 아니었는지:
if ai_removed_dates:
    display(rain_before(ai_removed_dates))


In [ ]:
# 3-3. 11/4 로거 교체 — 앞뒤 3일 중앙값 비교. 비는 왔나?
step = before_after(as_hourly(gw), "2025-11-04 10:00")
print("11/1–11/7 강수:", rain_series()["2025-11-01":"2025-11-07"].sum(), "mm")


In [ ]:
# 3-4. 내 최종 일 수위 my_daily — AI 코드를 고쳐서 만드세요(코드 2점은 이 칸을 봅니다).
# 조건: 날짜를 인덱스로 한 일 단위 Series · 같은 시각 정리 · 로거 오류 제거 · 11/4 단차 보정 · 근거 없는 긴 보간 금지
my_daily = None   # ← 여기에

if my_daily is not None:
    tab = xcorr(my_daily)
    display(tab)


## 4. 해석 (해석 2점) — 수치와 근거를 넣어 짧게

**Q1. 시간 축.** 원자료의 문제(같은 시각 몇 행, 빠진 시각 언제 몇 개, 빈칸 몇 개)와 AI가 각각을 어떻게 처리했는지. 7/7–7/18을 직선으로 이어도 되나? (근거: 그 기간 강수)

**Q2. 이상값.** AI가 이상값으로 지운 것 중 **실제 지하수위 변화**(큰 비 뒤 상승)였던 것과, AI가 **놓친 로거 오류**는? (근거: 3-2 표, 강수)

**Q3. 로거 교체.** 11/4 단차의 크기와 처리 — 지웠나, 보정했나? 왜 그렇게 해야 하나?

**Q4. 반응 지연.** 비가 온 뒤 며칠 만에 수위가 오르기 시작하나? '수위'와 '수위 변화량'의 교차상관 결과가 다르면 왜? 이 지연은 지질학적으로 무엇을 뜻하나?


## 5. AI 활용 기록지 (기록지 1점 — 2건 이상)

| # | 도구 | 프롬프트(요지) | 산출물(요지) | 검증(무엇으로 확인했나) | 수정(무엇을 고쳤나) |
|---|---|---|---|---|---|
| 1 |  |  |  |  |  |
| 2 |  |  |  |  |  |
